In [ ]:
!wget -q -c -O rev1-benchmark-data.tar.gz https://github.com/huboqiang/omnialigner/releases/download/rev1/rev1-benchmark-data.tar.gz
!git clone --quiet --depth 1 --branch rev1 https://github.com/huboqiang/omnialigner.git omnialigner-rev1

from pathlib import Path
import hashlib, tarfile, sys
assert (3, 10) <= sys.version_info[:2] < (3, 12), "Use a Python 3.10 or 3.11 Jupyter kernel."
archive = Path("rev1-benchmark-data.tar.gz").resolve()
checkout = Path("omnialigner-rev1").resolve()
assert archive.is_file() and (checkout / ".git").is_dir(), "Download or clone failed; inspect the messages above."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "9b98464f0b6625bae48514c601b740caee50aeb008525f103a362fc7a630d7d3", "Data archive checksum mismatch"
destination = checkout / "benchmarks/rev1"
with tarfile.open(archive, "r:gz") as bundle:
    for member in bundle.getmembers():
        target = (destination / member.name).resolve()
        assert target.is_relative_to(destination) and not (member.issym() or member.islnk()), "Unsafe archive member"
    bundle.extractall(destination)
PACKAGE_ROOT = destination / "NCS"
print("Downloaded and verified rev1 data; package:", PACKAGE_ROOT.name)


In [ ]:
%pip install -q -r {PACKAGE_ROOT / "requirements.txt"}
%pip install -q torch==2.2.0 --index-url https://download.pytorch.org/whl/cpu
%cd {PACKAGE_ROOT}


## Download and run
Use a **fresh Python 3.10/3.11 Jupyter kernel** with `wget` and `git` available. The first two commands download the public data archive and clone the `rev1` branch. No private directory or credentials are required. The six-dataset scoring run is CPU-based; allow 48–64 GB RAM for the largest dataset. The table-only and DLPFC examples are smaller.


# DLPFC worked example: weights → raw coordinates → aligned coordinates → six scores

This example loads the selected OmniAligner geometry parameters, applies the frozen coarse stack transform and learned nonlinear field to the **original input coordinates**, then scores all three adjacent DLPFC pairs. Saved selected coordinates and table values are used only for validation, never as the inference output.

The portable checkpoint contains the same learned control-point tensors and affine parameters as the selected checkpoint, plus the frozen raw-to-coarse frame transforms. It contains no `expected_xy` array. The coarse transforms were recovered from the saved pre-nonlinear stage, not from selected final coordinates. Single-precision replay is checked with an explicit coordinate tolerance of 0.001 benchmark-frame pixels; these are not physically calibrated micrometres. This is checkpoint inference, not a new agent/coarse-matching or optimizer run.

In [1]:
from pathlib import Path
import os, sys, json, csv, hashlib, importlib.util, importlib.metadata, warnings, time
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from IPython.display import display

def locate_repository():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks/OmniAligner_NCS_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Run from this repository or its notebooks directory")
REPO_ROOT = locate_repository()
DATA_ROOT = REPO_ROOT / "data"
INPUT_ROOT = DATA_ROOT
input_records = json.loads((INPUT_ROOT / "evaluation_inputs_manifest.json").read_text())
input_index = {(x["dataset"], x["layer"]): x for x in input_records}
OUTPUT_ROOT = REPO_ROOT / "results" / "weight_example"
DATASETS = ["DLPFC"]
VERIFY_SOURCE_HASHES = True
ATOL = 1e-10
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["PCC", "CosSim", "SSIM", "MI", "MSE", "Region"]
NATIVE_KEYS = ["PCC", "CosSim", "SSIM", "MI", "Expression_MSE_native_ASSD_bygrid", "Accuracy"]
manifest = json.loads((REPO_ROOT / "notebooks/OmniAligner_NCS_manifest.json").read_text())
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)
reference_path = REPO_ROOT / "notebooks/OmniAligner_NCS_reference.csv"
assert sha256(reference_path) == manifest["expected_table_sha256"]
reference = pd.read_csv(reference_path).set_index("Dataset")
print("Reference values (not recomputed yet)")
display(reference[METRICS].round(6))

Reference values (not recomputed yet)


,PCC,CosSim,SSIM,MI,MSE,Region
Dataset,,,,,,
DLPFC,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759
MHPR,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039
MMPC,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546
MPC,0.333290,0.625865,0.323609,0.152123,0.341752,0.978106
E14_16,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624
MLN,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042


In [2]:
helper_dir = REPO_ROOT / manifest["helper_dir"]
for name, expected_hash in manifest["helper_sha256"].items():
    assert sha256(helper_dir / name) == expected_hash, name
sys.path.insert(0, str(helper_dir))
import ncs_score as scoring
native_path = REPO_ROOT / manifest["native_scoring_file"]
assert sha256(native_path) == manifest["native_scoring_sha256"]
spec = importlib.util.spec_from_file_location("selected_ncs_native_metrics", native_path)
native = importlib.util.module_from_spec(spec)
spec.loader.exec_module(native)

inventory = []
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    assert sha256(DATA_ROOT / config["coordinates"]) == config["coordinates_sha256"], dataset
    for item in config["weights"]:
        assert sha256(DATA_ROOT / item["path"]) == item["sha256"], item["path"]
    for layer in range(len(config["sections"])):
        record = input_index[(dataset, layer)]
        source = INPUT_ROOT / record["path"]
        assert source.is_file(), source
        assert record["source_h5ad_sha256"] == config["source_sha256"][layer]
        if VERIFY_SOURCE_HASHES:
            assert sha256(source) == record["sha256"], source
    inventory.append({"Dataset": dataset, "Sections": len(config["sections"]), "Pairs": config["pairs"],
                      "Original cells": config["original_cells"], "Scored cells": config["evaluation_cells"],
                      "Gene arguments": len(config["evaluation_genes"]), "Unique genes": len(set(config["evaluation_genes"]))})
display(pd.DataFrame(inventory).set_index("Dataset"))
save_json(OUTPUT_ROOT / "INPUT_VALIDATION.json", {"source_hashes_checked": VERIFY_SOURCE_HASHES, "datasets": DATASETS, "inventory": inventory})

,Sections,Pairs,Original cells,Scored cells,Gene arguments,Unique genes
Dataset,,,,,,
DLPFC,4,3,14243,14243,123,117


In [3]:
import torch
from weight_inference import load_weights, align_raw_coordinates
weights = load_weights(DATA_ROOT / "weights/DLPFC/portable_geometry.pth")
assert "expected_xy" not in weights
assert weights["original_checkpoint_sha256"] == manifest["datasets"]["DLPFC"]["weights"][0]["sha256"]
print("Loaded selected geometry parameters; inference on CPU")
print("Checkpoint frame extent:", weights["max_size"])


Loaded selected geometry parameters; inference on CPU
Checkpoint frame extent: 1280


## Apply the saved transforms to original coordinates
No fitting to the final selected coordinates occurs here. The reference NPZ is opened only after inference to measure replay error.

In [4]:
aligned_by_layer = []
raw_by_layer = []
all_ids = []
for layer in range(4):
    record = input_index[("DLPFC", layer)]
    with np.load(INPUT_ROOT / record["path"], allow_pickle=False) as raw:
        original_xy = raw["raw_xy"].copy()
        ids = np.asarray([f"s{layer}_{value}" for value in raw["ids"]])
    predicted_xy, diagnostics = align_raw_coordinates(weights, layer, original_xy)
    assert np.isfinite(predicted_xy).all()
    aligned_by_layer.append(predicted_xy)
    raw_by_layer.append(original_xy)
    all_ids.extend(ids)
aligned_xy = np.concatenate(aligned_by_layer)
reference_coordinates = np.load(DATA_ROOT / manifest["datasets"]["DLPFC"]["coordinates"], allow_pickle=False)
index = {value: i for i, value in enumerate(reference_coordinates["ids"])}
reference_xy = reference_coordinates["nonrigid"][[index[value] for value in all_ids]]
max_coordinate_error = float(np.abs(aligned_xy - reference_xy).max())
assert max_coordinate_error <= 0.001
print("Maximum coordinate difference (benchmark pixels):", max_coordinate_error)
np.savez_compressed(OUTPUT_ROOT / "coordinates_from_weights.npz", ids=np.asarray(all_ids), layers=np.concatenate([np.full(len(x),i) for i,x in enumerate(aligned_by_layer)]), nonrigid=aligned_xy)


Maximum coordinate difference (benchmark pixels): 0.00048828125


## Inspect the transformation

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for layer, (before, after) in enumerate(zip(raw_by_layer, aligned_by_layer)):
    axes[0].scatter(before[:,0],before[:,1],s=1,alpha=.35,label=str(layer+1))
    axes[1].scatter(after[:,0],after[:,1],s=1,alpha=.35,label=str(layer+1))
axes[0].set_title("Original input coordinates")
axes[1].set_title("Coordinates computed from weights")
for ax in axes:
    ax.set_aspect("equal");ax.set_xlabel("x (input / benchmark frame)");ax.set_ylabel("y (input / benchmark frame)")
axes[1].legend(title="Section",markerscale=4)
fig.tight_layout()
from io import BytesIO
from IPython.display import Image
plot_buffer = BytesIO()
fig.savefig(plot_buffer, format="png", dpi=150, bbox_inches="tight")
display(Image(data=plot_buffer.getvalue()))
plt.close(fig)


## Score the newly computed coordinates
The fixed 123 gene arguments (117 distinct genes) and native six-metric scorer are identical to the full NCS replay. Every adjacent pair is included. Original expression values and labels come from the verified input pack.

In [6]:
def load_scoring_slice(dataset, layer, coordinates):
    config = manifest["datasets"][dataset]
    record = input_index[(dataset, layer)]
    with np.load(INPUT_ROOT / record["path"], allow_pickle=False) as raw:
        matrix = sp.csr_matrix((raw["data"], raw["indices"], raw["indptr"]), shape=tuple(raw["shape"]))
        source = ad.AnnData(X=matrix)
        source.obs_names = raw["ids"].astype(str)
        source.var_names = raw["genes"].astype(str)
        source.obs[config["label_column"]] = raw["labels"].astype(str)
        source.obsm["spatial"] = raw["raw_xy"]
    try:
        wanted = [f"s{layer}_{value}" for value in source.obs_names]
        selected = coordinates["layers"] == layer
        ids = coordinates["ids"][selected].tolist()
        assert len(set(ids)) == len(ids) == len(wanted)
        assert set(ids) == set(wanted), (dataset, layer, "Cell identity mismatch")
        lookup = {value: index for index, value in enumerate(ids)}
        order = np.asarray([lookup[value] for value in wanted])
        xy = coordinates["nonrigid"][selected][order]
        keep = coordinates["historical_cohort"][selected][order] if "historical_cohort" in coordinates.files else np.ones(source.n_obs, dtype=bool)
        assert np.isfinite(xy).all() and xy.shape == (source.n_obs, 2)
        unique_genes = list(dict.fromkeys(config["evaluation_genes"]))
        data = source[keep, unique_genes].copy()
        data.obsm["spatial"] = xy[keep]
        data.obs["Region"] = data.obs[config["label_column"]].astype(str)
        if not sp.issparse(data.X):
            data.X = sp.csr_matrix(data.X)
        return data
    finally:
        if source.isbacked:
            source.file.close()

In [7]:
pair_rows=[]
with np.load(OUTPUT_ROOT / "coordinates_from_weights.npz",allow_pickle=False) as coordinates:
    slices=[load_scoring_slice("DLPFC",i,coordinates) for i in range(4)]
    for i in range(3):
        values, gene_records, support = scoring.score_pair(native,slices[i],slices[i+1],manifest["datasets"]["DLPFC"]["evaluation_genes"])
        pair_rows.append({"pair_index":i,**values,**support})
means = {name:scoring.c.strict_mean([row[key] for row in pair_rows],3) for name,key in zip(METRICS,NATIVE_KEYS)}
actual = np.array([means[name] for name in METRICS])
expected = reference.loc["DLPFC",METRICS].to_numpy(dtype=float)
np.testing.assert_allclose(actual,expected,atol=1e-10,rtol=0)
display(pd.DataFrame({"From weights":actual,"Current table":expected,"Difference":actual-expected},index=METRICS))
pd.DataFrame(pair_rows).to_csv(OUTPUT_ROOT / "pair_metrics.csv",index=False)
result={"all_six_scores_match":True,"pairs":3,"max_coordinate_difference":max_coordinate_error,"coordinate_tolerance":0.001,"score_tolerance":1e-10,"maximum_score_difference":float(np.abs(actual-expected).max()),"reads_selected_coordinates_as_prediction":False,"coarse_stage":"frozen raw-to-coarse affine transforms"}
save_json(OUTPUT_ROOT / "VALIDATION.json",result)
print(json.dumps(result,indent=2))


,From weights,Current table,Difference
PCC,0.547930,0.547930,0.000000e+00
CosSim,0.783503,0.783503,0.000000e+00
SSIM,0.531970,0.531970,0.000000e+00
MI,0.299724,0.299724,0.000000e+00
MSE,0.127579,0.127579,-2.775558e-17
Region,0.884759,0.884759,0.000000e+00


{
  "all_six_scores_match": true,
  "pairs": 3,
  "max_coordinate_difference": 0.00048828125,
  "coordinate_tolerance": 0.001,
  "score_tolerance": 1e-10,
  "maximum_score_difference": 2.7755575615628914e-17,
  "reads_selected_coordinates_as_prediction": false,
  "coarse_stage": "frozen raw-to-coarse affine transforms"
}
